# A2 - Planification nutritionnelle (DIET Problem)

Le Diet Problem est un classique de la recherche operationnelle : determiner la combinaison d'aliments la moins coutee satisfaisant l'ensemble des besoins nutritionnels journaliers. Modelisable comme un programme lineaire en nombres entiers (Knapsack multi-dimensionnel), il se prete remarquablement a une modelisation CP-SAT avec des contraintes sur les calories, proteines, lipides, glucides, vitamines et mineraux. Le sujet s'etend naturellement vers la generation de menus hebdomadaires equilibres en ajoutant des contraintes de variete (ne pas manger le meme plat deux jours de suite), de saisonnalite, de budget, et de preferences alimentaires.

## Objectifs
- Modeliser le Diet Problem comme un knapsack multi-dimensionnel avec CP-SAT (variables binaires par aliment, contraintes nutritionnelles)
- Etendre le modele en planification de menus hebdomadaires avec contraintes de variete, budget et saisonnalite
- Integrer des preferences utilisateur comme soft constraints avec penalites ponderees
Benchmarker sur les donnees USDA FoodData Central et les apports nutritionnels OMS/ANSES
Comparer l'approche CP-SAT avec une resolution PLNE classique (OR-Tools linear solver)

| Ligne de code | Équivalent mathématique |
|---|---|
| `solver = CreateSolver("GLOP")` | "Je vais utiliser l'algorithme du simplexe" |
| `x1 = NumVar(0, inf, "pain")` | $x_1 \geq 0$ (variable continue) |
| `solver.Add(250*x1 + 150*x2 >= 1000)` | La contrainte $250 x_1 + 150 x_2 \geq 1000$ |
| `solver.Minimize(x1 + 0.5*x2)` | $\min \; x_1 + 0{,}5 \, x_2$ |
| `solver.Solve()` | "Vas-y, trouve l'optimum" |

In [38]:
import csv
import os
from ortools.linear_solver import pywraplp
from ortools.sat.python import cp_model

In [39]:
FOODS = [
    # Féculents
    ("pates_cuites",      3, 130,  5,  1, 25,  2, 400),
    ("riz_cuit",          3, 130,  3,  0, 28,  0, 400),
    ("pain",             30, 250,  9,  1, 48,  3, 300),
    ("pommes_de_terre",  15,  85,  2,  0, 20,  2, 400),
    ("flocons_avoine",   20, 380, 13,  7, 60, 10, 150),
    ("semoule_cuite",    10, 110,  4,  0, 23,  1, 400),

    # Légumineuses
    ("lentilles",        15, 115,  9,  0, 20,  8, 400),
    ("pois_chiches",     20, 165,  9,  3, 27,  8, 400),
    ("haricots_rouges",  20, 125,  9,  0, 22,  6, 400),

    # Protéines animales
    ("oeufs",            30, 155, 13, 11,  1,  0, 300),
    ("poulet",          120, 165, 31,  4,  0,  0, 300),
    ("thon_boite",       80, 130, 27,  1,  0,  0, 200),
    ("sardines",         60, 200, 25, 11,  0,  0, 200),
    ("jambon",          100, 115, 18,  4,  1,  0, 200),

    # Produits laitiers
    ("lait",             10,  46,  3,  2,  5,  0, 500),
    ("yaourt",           15,  55,  4,  1,  5,  0, 500),
    ("fromage_blanc",    25,  75,  8,  3,  4,  0, 300),
    ("emmental",        100, 380, 28, 30,  0,  0, 100),

    # Légumes
    ("brocoli",          30,  35,  3,  0,  7,  3, 500),
    ("carottes",         15,  40,  1,  0, 10,  3, 500),
    ("tomate",           30,  18,  1,  0,  4,  1, 500),
    ("epinards",         40,  25,  3,  0,  4,  2, 400),
    ("courgette",        20,  17,  1,  0,  3,  1, 500),
    ("champignons",      50,  22,  3,  0,  3,  1, 300),

    # Fruits
    ("pomme",            30,  52,  0,  0, 14,  2, 400),
    ("banane",           25,  89,  1,  0, 23,  3, 400),
    ("orange",           25,  47,  1,  0, 12,  2, 400),

    # Matières grasses
    ("huile_olive",     150, 884,  0,100,  0,  0,  50),
    ("beurre",          100, 717,  1, 81,  0,  0,  50),
    ("amandes",         250, 580, 21, 50, 22, 12,  80),
]

# nom, min, max
NUTRIENTS = [
    ("kcal",    2000, 2500),
    ("prot",      60,  150),
    ("lip",       50,   90),
    ("gluc",     200,  350),
    ("fibres",    25, None),
]

In [ ]:



def solve_diet(foods, nutrients):
    """Minimise le coût en respectant les contraintes nutritionnelles."""
    model = cp_model.CpModel()
    quantities = [model.NewIntVar(0, food[-1], food[0]) for food in foods] # NewIntVar(0,  max_unites, name)

    # Contraintes nutritionnelles
    for j, (_, lo, hi) in enumerate(nutrients):
        total = sum(foods[i][j + 2] * quantities[i] for i in range(len(foods))) # Créer l'équation feature_n * x1 + feature_n * x2 ... + feature_n * x_n
        model.Add(total >= lo * 100)
        if hi is not None:
            model.Add(total <= hi  * 100)

    # Minimise le prix
    model.Minimize(sum(foods[i][1] * quantities[i] for i in range(len(foods))))

    solver = cp_model.CpSolver()
    status = solver.Solve(model)
    return status, solver, quantities

def print_solution(foods, nutrients, status, solver, quantities):
    """Affiche la solution et renvoie les indices des aliments utilisés."""
    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print("Pas de solution réalisable.")
        return []

    if status == cp_model.FEASIBLE:
        print("Solution trouvée mais optimalité non prouvée.")

    print(f"Coût total : {solver.ObjectiveValue() / 10000:.2f} €\n")

    print("Quantités :")
    used = []
    for i, food in enumerate(foods):
        qty = solver.Value(quantities[i])
        if qty > 0:
            used.append(i)
            # qty est en unités de 10 g
            print(f"  {food[0]:15s} : {qty:3d} unités  ({qty * 10:4d} g)")

    print("\nApports atteints :")
    for j, (name, lo, hi) in enumerate(nutrients):
        total = sum(foods[i][j + 2] * solver.Value(quantities[i])
                    for i in range(len(foods))) // 100
        hi_str = f" - {hi}" if hi else ""
        print(f"  {name:8s} : {total:5d}  (cible: {lo}{hi_str})")

    return used

In [42]:
def main():
    print("=== Repas 1 ===")
    status, solver, qty = solve_diet(FOODS, NUTRIENTS)
    used = print_solution(FOODS, NUTRIENTS, status, solver, qty)

    # Exclure les aliments déjà utilisés pour le repas suivant
    remaining = [f for i, f in enumerate(FOODS) if i not in used]

    print("\n=== Repas 2 ===")
    status, solver, qty = solve_diet(remaining, NUTRIENTS)
    print_solution(remaining, NUTRIENTS, status, solver, qty)


if __name__ == "__main__":
    main()

=== Repas 1 ===
Coût total : 1.10 €

Quantités :
  pates_cuites    : 400 unités  (4000 g)
  riz_cuit        : 400 unités  (4000 g)
  flocons_avoine  : 150 unités  (1500 g)
  lentilles       :  45 unités  ( 450 g)
  pois_chiches    :   6 unités  (  60 g)
  oeufs           :  27 unités  ( 270 g)
  beurre          :  40 unités  ( 400 g)

Apports atteints :
  kcal     :  2000  (cible: 2000 - 2500)
  prot     :    60  (cible: 60 - 150)
  lip      :    50  (cible: 50 - 90)
  gluc     :   312  (cible: 200 - 350)
  fibres   :    27  (cible: 25)

=== Repas 2 ===
Coût total : 2.64 €

Quantités :
  pain            : 300 unités  (3000 g)
  pommes_de_terre :   1 unités  (  10 g)
  semoule_cuite   : 400 unités  (4000 g)
  haricots_rouges : 315 unités  (3150 g)
  huile_olive     :  47 unités  ( 470 g)

Apports atteints :
  kcal     :  2000  (cible: 2000 - 2500)
  prot     :    71  (cible: 60 - 150)
  lip      :    50  (cible: 50 - 90)
  gluc     :   305  (cible: 200 - 350)
  fibres   :    31  (cible:

In [5]:
import pandas as pd
df = pd.read_csv("ciqual_clean.csv")

In [6]:
df.columns

Index(['code', 'nom', 'groupe', 'sous_groupe', 'kcal', 'prot_g', 'lip_g',
       'gluc_g', 'fibres_g', 'sel_g', 'calcium_mg', 'fer_mg', 'prix_cts_100g',
       'min_g', 'max_g'],
      dtype='str')

In [8]:
df["groupe"].unique()

<ArrowStringArray>
[                  'entrées et plats composés',
 'fruits, légumes, légumineuses et oléagineux',
                         'produits céréaliers',
                    'viandes, oeufs, poissons',
                           'produits laitiers']
Length: 5, dtype: str

In [10]:
df["sous_groupe"].unique()


<ArrowStringArray>
[                    'salades composées et crudités',
                                            'soupes',
                                    'plats composés',
                   'pizzas, tartes et crêpes salées',
                                         'sandwichs',
                     'feuilletées et autres entrées',
                                           'légumes',
              'pommes de terre et autres tubercules',
                                      'légumineuses',
                                            'fruits',
                            'pâtes, riz et céréales',
                                     'pâtes à tarte',
                                    'viandes cuites',
                  'autres produits à base de viande',
                                    'poissons cuits',
                                     'poissons crus',
                     'mollusques et crustacés cuits',
 'produits à base de poissons et produits de la mer',
         

In [50]:
df["alim_ssssgrp_nom_fr"].unique()


<StringArray>
[                                           nan,
                                            '-',
               'plats de viande sans garniture',
                 'plats de viande et féculents',
      'plats de viande et légumes/légumineuses',
              'plats de poisson sans garniture',
                'plats de poisson et féculents',
                'plats de légumes/légumineuses',
                      'plats de céréales/pâtes',
                             'plats de fromage',
                     'autres plats végétariens',
                                 'légumes crus',
                                'légumes cuits',
                'légumes séchés ou déshydratés',
   'légumes et leurs produits de la Martinique',
      'légumes et leurs produits de la Réunion',
                          'légumineuses cuites',
                        'légumineuses fraîches',
                          'légumineuses sèches',
                                  'fruits crus',
      